<a href="https://colab.research.google.com/github/danielalopez-lang/Tesis_Maestria-Resistoma_Chakras-Tena-/blob/main/Analisis_Estadistico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# ==============================================================================
# 1. CONFIGURACIÓN Y CARGA DE DATOS
# ==============================================================================
# Nombre del archivo Excel previamente subido a Colab
nombre_archivo = 'Pruebas_Estadísticas_TESIS.xlsx'

try:
    # Cargar la hoja de cálculo
    df = pd.read_excel(nombre_archivo)
    print("Archivo cargado correctamente en Google Colab.\n")
except Exception as e:
    print(f"Error al cargar el archivo. Verifica que lo subiste con el nombre correcto: {e}")

# ==============================================================================
# 2. DEFINICIÓN DE COLUMNAS Y MÉTRICAS DESCRIPTIVAS
# ==============================================================================
columna_grupo = 'Sitio'
variables = ['pH', '% Humedad', '% COS']

# Lista de agregaciones estadísticas a calcular por cada variable
lista_metricas = [
    ('Media', 'mean'),
    ('Desv_Estd', 'std'),
    ('Mediana', 'median'),
    ('Q1', lambda x: x.quantile(0.25)),
    ('Q3', lambda x: x.quantile(0.75)),
    ('IQR', lambda x: x.quantile(0.75) - x.quantile(0.25)),
    ('Min', 'min'),
    ('Max', 'max')
]

# ==============================================================================
# 3. PROCESAMIENTO Y CÁLCULO
# ==============================================================================
# Agrupar por Sitio y aplicar la lista de métricas a pH, Humedad y COS
resumen_estadistico = df.groupby(columna_grupo)[variables].agg(lista_metricas).round(2)

# ==============================================================================
# 4. PRESENTACIÓN Y EXPORTACIÓN DE RESULTADOS
# ==============================================================================
print("=" * 80)
print("RESUMEN DE ESTADÍSTICA DESCRIPTIVA POR SITIO")
print("=" * 80)

# Mostrar la tabla en formato ordenado en Colab
display(resumen_estadistico)

# Guardar la tabla resumen en un archivo Excel descargable
archivo_salida = 'Resultado_Estadistica_Descriptiva.xlsx'
resumen_estadistico.to_excel(archivo_salida)
print(f"\nArchivo exportado exitosamente como: '{archivo_salida}'")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==============================================================================
# 2. CONFIGURACIÓN DE VARIABLES
# ==============================================================================
columna_sitio = 'Sitio'
variables = ['pH', '% Humedad', '% COS']

# Configurar el estilo base de los gráficos para que se vean elegantes
sns.set_theme(style="whitegrid")

# ==============================================================================
# 3. GENERACIÓN DE DIAGRAMAS DE CAJA
# ==============================================================================
# Bucle para crear un gráfico por cada variable numérica
for var in variables:
    # Crear un lienzo para cada gráfico (tamaño 8x6 pulgadas)
    plt.figure(figsize=(8, 6))

    # Crear el diagrama de caja
    # Nota: seaborn usa los estadísticos base de matplotlib, que calculan
    # los cuartiles usando el método lineal (mediana inclusiva por defecto).
    ax = sns.boxplot(
        data=df,
        x=columna_sitio,
        y=var,
        showmeans=True, # Habilita los marcadores de la media
        meanprops={
            "marker": "D",                # 'D' de Diamante para la media
            "markerfacecolor": "white",   # Relleno blanco
            "markeredgecolor": "black",   # Borde negro
            "markersize": 7               # Tamaño del marcador
        },
        flierprops={
            "marker": "o",                # 'o' circular para puntos atípicos
            "markerfacecolor": "red",     # Color rojo para resaltarlos
            "markeredgecolor": "black",
            "markersize": 6
        },
        palette="pastel" # Colores suaves para las cajas
    )

    # Configuración de títulos y ejes según tus requerimientos
    plt.title(f'Distribución de {var} según el sitio de muestreo', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Sitio de muestreo', fontsize=12, labelpad=10)
    plt.ylabel(var, fontsize=12, labelpad=10)

    # Ajustar los márgenes automáticamente
    plt.tight_layout()

    # Guardar cada gráfico automáticamente en alta calidad
    nombre_imagen = f'Boxplot_{var}.png'
    plt.savefig(nombre_imagen, dpi=300)

    # Mostrar el gráfico en la pantalla de Colab
    plt.show()

print("Todos los diagramas de caja se han generado y guardado como imágenes PNG")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

# ==============================================================================
# 1. CARGA DE DATOS Y CONFIGURACIÓN INICIAL
# ==============================================================================
nombre_archivo = 'Pruebas_Estadísticas_TESIS.xlsx'

try:
    df = pd.read_excel(nombre_archivo)
    print("Archivo cargado correctamente para la evaluación de supuestos.\n")
except Exception as e:
    print(f"Error al cargar el archivo: {e}")

# Limpieza de espacios en blanco en los nombres de variables
df.columns = df.columns.str.strip()

columna_sitio = 'Sitio'
variables = ['pH', '% Humedad', '% COS']

# Estilo visual de gráficos
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# Listas para almacenar los resultados en formato tabular
resultados_shapiro = []
resultados_bf = []
resumen_decision = []

# Copia del dataframe original para adjuntar los residuos calculados
df_residuos_tabla = df.copy()

# ==============================================================================
# 2. CÁLCULOS ESTADÍSTICOS Y GENERACIÓN DE GRÁFICOS
# ==============================================================================
for var in variables:
    # --------------------------------------------------------------------------
    # A) Cálculo de Residuos por Chakra/Sitio
    # --------------------------------------------------------------------------
    formula = f"Q('{var}') ~ C({columna_sitio})"
    modelo = ols(formula, data=df).fit()

    col_residuo_nombre = f'Residuo_{var}'
    df_residuos_tabla[col_residuo_nombre] = modelo.resid.round(4)
    residuos = modelo.resid

    # --------------------------------------------------------------------------
    # B) Gráfico Q-Q de Residuos con Línea de Tendencia Lineal
    # --------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(7, 5))

    # line='s' genera la línea de tendencia lineal estandarizada de 45°
    sm.qqplot(
        residuos,
        line='s',
        ax=ax,
        marker='o',
        markerfacecolor='#2b5c8f',
        markeredgecolor='black',
        alpha=0.85
    )

    # Títulos y etiquetas requeridas
    ax.set_title(f'Gráfico Q-Q de Residuos: {var}', fontsize=13, fontweight='bold', pad=12)
    ax.set_xlabel('Valores normales teóricos', fontsize=11, labelpad=8)
    ax.set_ylabel('Residuos ordenados', fontsize=11, labelpad=8)

    plt.tight_layout()
    nombre_img = f"QQ_Plot_{var.replace('%', 'Pct').replace(' ', '_')}.png"
    plt.savefig(nombre_img, dpi=300)
    plt.show()

    # --------------------------------------------------------------------------
    # C) Prueba de Normalidad (Shapiro-Wilk sobre residuos)
    # --------------------------------------------------------------------------
    stat_sw, p_sw = stats.shapiro(residuos)
    cumple_norm = p_sw > 0.05

    resultados_shapiro.append({
        'Variable': var,
        'Estadístico W': round(stat_sw, 4),
        'p-valor': round(p_sw, 4),
        'Normalidad (p > 0.05)': 'Cumple' if cumple_norm else 'No cumple'
    })

    # --------------------------------------------------------------------------
    # D) Prueba de Homocedasticidad (Brown-Forsythe basada en la mediana)
    # --------------------------------------------------------------------------
    grupos_datos = [grupo[var].dropna().values for _, grupo in df.groupby(columna_sitio)]
    stat_bf, p_bf = stats.levene(*grupos_datos, center='median')
    cumple_homo = p_bf > 0.05

    resultados_bf.append({
        'Variable': var,
        'Estadístico F': round(stat_bf, 4),
        'p-valor': round(p_bf, 4),
        'Homocedasticidad (p > 0.05)': 'Cumple' if cumple_homo else 'No cumple'
    })

    # --------------------------------------------------------------------------
    # E) Decisión del Test Estadístico
    # --------------------------------------------------------------------------
    test_recomendado = 'ANOVA de un factor' if (cumple_norm and cumple_homo) else 'Kruskal-Wallis'

    resumen_decision.append({
        'Variable': var,
        'Prueba Normalidad (SW)': 'Cumple' if cumple_norm else 'No cumple',
        'Prueba Homocedasticidad (BF)': 'Cumple' if cumple_homo else 'No cumple',
        'Prueba Recomendada': test_recomendado
    })

# ==============================================================================
# 3. CREACIÓN Y PRESENTACIÓN DE TABLAS DE RESULTADOS
# ==============================================================================
tabla_shapiro = pd.DataFrame(resultados_shapiro)
tabla_bf = pd.DataFrame(resultados_bf)
tabla_resumen = pd.DataFrame(resumen_decision)

print("=" * 80)
print("TABLA 1: CÁLCULO DE RESIDUOS POR CHAKRA/SITIO (MUESTRA PARCIAL)")
print("=" * 80)
display(df_residuos_tabla.head(10))

print("\n" + "=" * 80)
print("TABLA 2: PRUEBA DE NORMALIDAD DE SHAPIRO-WILK (EN RESIDUOS)")
print("=" * 80)
display(tabla_shapiro)

print("\n" + "=" * 80)
print("TABLA 3: PRUEBA DE HOMOCEDASTICIDAD DE BROWN-FORSYTHE (MEDIANA)")
print("=" * 80)
display(tabla_bf)

print("\n" + "=" * 80)
print("TABLA 4: CUADRO RESUMEN DE DECISIÓN ESTADÍSTICA")
print("=" * 80)
display(tabla_resumen)

# ==============================================================================
# 4. EXPORTACIÓN A EXCEL
# ==============================================================================
archivo_excel_salida = 'Resultados_Evaluacion_Supuestos.xlsx'

with pd.ExcelWriter(archivo_excel_salida) as writer:
    df_residuos_tabla.to_excel(writer, sheet_name='Residuos_Calculados', index=False)
    tabla_shapiro.to_excel(writer, sheet_name='Shapiro_Wilk', index=False)
    tabla_bf.to_excel(writer, sheet_name='Brown_Forsythe', index=False)
    tabla_resumen.to_excel(writer, sheet_name='Resumen_Decision', index=False)

print(f"\n Todas las tablas han sido guardadas en el archivo Excel: '{archivo_excel_salida}'")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# ==============================================================================
# 1. CARGA Y PREPARACIÓN DE DATOS
# ==============================================================================
nombre_archivo = 'Pruebas_Estadísticas_TESIS.xlsx'

try:
    df = pd.read_excel(nombre_archivo)
    print("Archivo cargado correctamente para el análisis.\n")
except Exception as e:
    print(f"Error al cargar el archivo Excel: {e}")

# Limpiar espacios en blanco en los nombres de las columnas
df.columns = df.columns.str.strip()

columna_sitio = 'Sitio'
variables = ['pH', '% Humedad', '% COS']
alfa = 0.05

# Diccionarios para almacenar las tablas finales
tablas_anova = {}
tablas_tukey = {}

# ==============================================================================
# 2. PROCESAMIENTO: ANOVA Y TUKEY POR VARIABLE
# ==============================================================================
for var in variables:
    print("=" * 80)
    print(f" ANÁLISIS PARA LA VARIABLE: {var}")
    print("=" * 80)

    # --------------------------------------------------------------------------
    # A) ANOVA DE UN FACTOR
    # --------------------------------------------------------------------------
    formula = f"Q('{var}') ~ C({columna_sitio})"
    modelo = ols(formula, data=df).fit()
    tabla_anova_bruta = sm.stats.anova_lm(modelo, typ=1)

    # Extracción de métricas entre grupos y dentro de grupos
    ss_entre = tabla_anova_bruta.loc[f'C({columna_sitio})', 'sum_sq']
    df_entre = int(tabla_anova_bruta.loc[f'C({columna_sitio})', 'df'])
    ms_entre = tabla_anova_bruta.loc[f'C({columna_sitio})', 'mean_sq']
    f_stat = tabla_anova_bruta.loc[f'C({columna_sitio})', 'F']
    p_valor_anova = tabla_anova_bruta.loc[f'C({columna_sitio})', 'PR(>F)']

    ss_dentro = tabla_anova_bruta.loc['Residual', 'sum_sq']
    df_dentro = int(tabla_anova_bruta.loc['Residual', 'df'])
    ms_dentro = tabla_anova_bruta.loc['Residual', 'mean_sq']

    # Formatear Tabla ANOVA para la variable
    df_anova_var = pd.DataFrame([
        {
            'Fuente de Variación': 'Entre grupos (Sitios)',
            'Suma de Cuadrados (SS)': round(ss_entre, 4),
            'Grados de Libertad (GL)': df_entre,
            'Cuadrado Medio (MS)': round(ms_entre, 4),
            'Estadístico F': round(f_stat, 4),
            'p-valor': round(p_valor_anova, 4),
            'Significativo (α = 0.05)': 'Sí' if p_valor_anova < alfa else 'No'
        },
        {
            'Fuente de Variación': 'Dentro de los grupos (Error)',
            'Suma de Cuadrados (SS)': round(ss_dentro, 4),
            'Grados de Libertad (GL)': df_dentro,
            'Cuadrado Medio (MS)': round(ms_dentro, 4),
            'Estadístico F': '-',
            'p-valor': '-',
            'Significativo (α = 0.05)': '-'
        }
    ])

    tablas_anova[var] = df_anova_var

    print(f"\n TABLA ANOVA - {var}:")
    display(df_anova_var)

    # --------------------------------------------------------------------------
    # B) PRUEBA POST-HOC DE TUKEY HSD (SOLO SI ES SIGNIFICATIVO)
    # --------------------------------------------------------------------------
    if p_valor_anova < alfa:
        print(f"\n La variable '{var}' presenta diferencias significativas (p = {p_valor_anova:.4f} < {alfa}).")
        print(f" Ejecutando la prueba de Tukey HSD...")

        # Eliminar valores nulos para la prueba
        df_limpio = df[[columna_sitio, var]].dropna()

        tukey_res = pairwise_tukeyhsd(
            endog=df_limpio[var],
            groups=df_limpio[columna_sitio],
            alpha=alfa
        )

        # Convertir los resultados de Tukey a DataFrame de Pandas
        df_tukey_var = pd.DataFrame(
            data=tukey_res._results_table.data[1:],
            columns=tukey_res._results_table.data[0]
        )

        # Renombrar columnas para mayor claridad en español
        df_tukey_var.columns = [
            'Grupo 1', 'Grupo 2', 'Diferencia de Medias',
            'p-adj', 'Límite Inf (95%)', 'Límite Sup (95%)', 'Decisión H0'
        ]

        # Redondear valores numéricos
        df_tukey_var['Diferencia de Medias'] = df_tukey_var['Diferencia de Medias'].round(4)
        df_tukey_var['p-adj'] = df_tukey_var['p-adj'].round(4)
        df_tukey_var['Límite Inf (95%)'] = df_tukey_var['Límite Inf (95%)'].round(4)
        df_tukey_var['Límite Sup (95%)'] = df_tukey_var['Límite Sup (95%)'].round(4)

        # Mapeo explicativo en lugar de True/False
        df_tukey_var['Decisión H0'] = df_tukey_var['Decisión H0'].map({
            True: 'Se rechaza la H0',
            False: 'No se rechaza la H0'
        })

        tablas_tukey[var] = df_tukey_var

        print(f"\n TABLA PRUEBA DE TUKEY - {var}:")
        display(df_tukey_var)
    else:
        print(f"\nℹ La variable '{var}' NO presenta diferencias significativas (p = {p_valor_anova:.4f} >= {alfa}).")
        print(" No se aplica la prueba de Tukey.")

    print("\n" + "=" * 80 + "\n")

# ==============================================================================
# 3. EXPORTACIÓN DE RESULTADOS A EXCEL
# ==============================================================================
archivo_excel_salida = 'Resultados_ANOVA_y_Tukey.xlsx'

with pd.ExcelWriter(archivo_excel_salida) as writer:
    # Guardar tablas ANOVA
    for var, tabla in tablas_anova.items():
        nombre_hoja = f"ANOVA_{var.replace('%', 'Pct').replace(' ', '_')}"
        tabla.to_excel(writer, sheet_name=nombre_hoja[:31], index=False)

    # Guardar tablas Tukey
    for var, tabla in tablas_tukey.items():
        nombre_hoja = f"Tukey_{var.replace('%', 'Pct').replace(' ', '_')}"
        tabla.to_excel(writer, sheet_name=nombre_hoja[:31], index=False)

print(f" Todas las tablas de ANOVA y Tukey han sido exportadas a: '{archivo_excel_salida}'")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# ==============================================================================
# 1. CARGA DE DATOS
# ==============================================================================
nombre_archivo = 'Pruebas_Estadísticas_TESIS.xlsx'

try:
    df = pd.read_excel(nombre_archivo)
    print("Archivo cargado correctamente.\n")
except Exception as e:
    print(f"Error al cargar el archivo: {e}")

# Limpiar posibles espacios en blanco en los nombres de columnas
df.columns = df.columns.str.strip()

columna_sitio = 'Sitio'
variables = ['pH', '% Humedad', '% COS']
alfa = 0.05

# Configurar el estilo base del gráfico
sns.set_theme(style="whitegrid")

# ==============================================================================
# 2. GENERACIÓN DE BOXPLOTS CON LETRAS DE TUKEY
# ==============================================================================
for var in variables:
    # A) Verificar si existe diferencia significativa con ANOVA
    formula = f"Q('{var}') ~ C({columna_sitio})"
    modelo = ols(formula, data=df).fit()
    tabla_anova = sm.stats.anova_lm(modelo, typ=1)
    p_valor_anova = tabla_anova.loc[f'C({columna_sitio})', 'PR(>F)']

    # Mapeo por defecto de letras (vacío si no hay ANOVA significativo)
    letras_dict = {sitio: '' for sitio in df[columna_sitio].unique()}

    if p_valor_anova < alfa:
        # B) Ejecutar Tukey HSD y obtener las letras compactas
        df_limpio = df[[columna_sitio, var]].dropna()
        tukey = pairwise_tukeyhsd(endog=df_limpio[var], groups=df_limpio[columna_sitio], alpha=alfa)

        # Generar las letras asignadas a cada grupo
        # plot_simultaneous asigna letras de rango a la tabla de agrupamiento
        res_tabla = tukey._results_table

        # Algoritmo de asignación manual de grupos de letras Tukey
        # Agrupamos por media para ordenar correctamente las letras (a, b, c...)
        medias = df_limpio.groupby(columna_sitio)[var].mean().sort_values(ascending=False)
        sitios_ordenados = medias.index.tolist()

        # Extraer diferencias significativas entre parejas
        tukey_df = pd.DataFrame(data=tukey._results_table.data[1:], columns=tukey._results_table.data[0])

        # Asignación simplificada de letras de Tukey basada en diferencias
        # Crear matriz de no-diferencia
        n_grupos = len(sitios_ordenados)
        letras_asignadas = {sitio: '' for sitio in sitios_ordenados}

        # Construcción de letras compactas de significancia
        # Para visualización directa en biotecnología
        from statsmodels.stats.multicomp import MultiComparison
        mc = MultiComparison(df_limpio[var], df_limpio[columna_sitio])
        res_mc = mc.tukeyhsd()

        # Usamos una representación por rangos de diferencia de medias
        # Si un par no se rechaza, comparten letra
        # Para asignación estándar automatizada:
        import scipy.stats as stats

        # Calcular los máximos para ubicar la letra sobre el bigote/outlier
        maximos = df_limpio.groupby(columna_sitio)[var].max()

        # Construcción de mapeo de letras según Tukey
        # Usaremos el resultado del test de Tukey para construir las letras:

        # Creamos una lista con las letras del abecedario
        import string
        letras_lista = list(string.ascii_lowercase)

        # Matriz de diferencias
        sitios_unicos = df_limpio[columna_sitio].unique()
        matriz_dif = pd.DataFrame(False, index=sitios_unicos, columns=sitios_unicos)

        for _, fila in tukey_df.iterrows():
            g1, g2, rechazo = fila['group1'], fila['group2'], fila['reject']
            matriz_dif.loc[g1, g2] = rechazo
            matriz_dif.loc[g2, g1] = rechazo

        # Asignar letras según las comparaciones
        # Algoritmo simple de asignación de letras de significancia
        grupos_letras = {}
        letra_idx = 0

        for s1 in sitios_ordenados:
            if s1 not in grupos_letras:
                grupos_letras[s1] = letras_lista[letra_idx]
                for s2 in sitios_ordenados:
                    if s1 != s2 and not matriz_dif.loc[s1, s2]:
                        grupos_letras[s2] = grupos_letras.get(s2, '') + letras_lista[letra_idx]
                letra_idx += 1

        letras_dict = grupos_letras

    # C) Graficar el Boxplot
    plt.figure(figsize=(8, 6))

    ax = sns.boxplot(
        data=df,
        x=columna_sitio,
        y=var,
        showmeans=True,
        meanprops={
            "marker": "D",
            "markerfacecolor": "white",
            "markeredgecolor": "black",
            "markersize": 7
        },
        flierprops={
            "marker": "o",
            "markerfacecolor": "red",
            "markeredgecolor": "black",
            "markersize": 6
        },
        palette="pastel"
    )

    # D) Añadir las letras de Tukey en la parte superior de cada caja
    # Calculamos la posición Y para que la letra quede flotando arriba
    max_y_var = df[var].max()
    min_y_var = df[var].min()
    rango_y = max_y_var - min_y_var
    offset = rango_y * 0.05  # Separación del 5% del rango total

    # Obtener el orden de las categorías en el eje X
    sitios_eje_x = [tick.get_text() for tick in ax.get_xticklabels()]

    for i, sitio in enumerate(sitios_eje_x):
        if sitio in letras_dict and letras_dict[sitio] != '':
            # Obtener el valor máximo registrado para ese sitio (para colocar la letra arriba)
            val_max_sitio = df[df[columna_sitio] == sitio][var].max()
            letra = letras_dict[sitio]

            # Imprimir la letra en el gráfico
            ax.text(
                x=i,
                y=val_max_sitio + offset,
                s=letra,
                horizontalalignment='center',
                size=12,
                color='black',
                weight='bold'
            )

    # E) Personalizar etiquetas y títulos exactos
    plt.title(f'Distribución de {var} según el sitio de muestreo', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('sitio de muestreo', fontsize=12, labelpad=10)
    plt.ylabel(var, fontsize=12, labelpad=10)

    # Expandir un poco el límite superior de Y para que la letra no quede cortada
    plt.ylim(top=max_y_var + (rango_y * 0.12))

    plt.tight_layout()

    # Guardar gráfico
    nombre_img = f"Boxplot_Tukey_{var.replace('%', 'Pct').replace(' ', '_')}.png"
    plt.savefig(nombre_img, dpi=300)
    plt.show()

print("Gráficos con letras de significancia de Tukey generados y guardados con éxito")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

# ==============================================================================
# 1. CARGA Y PREPARACIÓN DE DATOS
# ==============================================================================
nombre_archivo = 'Pruebas_Estadísticas_TESIS.xlsx'

try:
    df = pd.read_excel(nombre_archivo)
    print("Archivo cargado correctamente.\n")
except Exception as e:
    print(f"Error al cargar el archivo Excel: {e}")

# Limpieza de espacios en blanco en los nombres de columnas
df.columns = df.columns.str.strip()

# Identificar columna de etiquetas de Chakra
columna_chakra = 'Chakra' if 'Chakra' in df.columns else 'Sitio'

# Limpiar columna Control_UFC y reemplazar 'IN' por NaN
df['Control_UFC'] = df['Control_UFC'].astype(str).str.strip()
df['Control_UFC_Limpio'] = pd.to_numeric(df['Control_UFC'].replace(['IN', 'In', 'in'], np.nan), errors='coerce')

# Transformación logarítmica: log10(UFC/g + 1)
df['Log_Control_UFC'] = np.log10(df['Control_UFC_Limpio'] + 1)

# Configuración de variables y títulos personalizados
config_variables = [
    {
        'columna': 'pH',
        'eje_x': 'pH',
        'titulo': 'Relación entre el pH y la abundancia bacteriana cultivable'
    },
    {
        'columna': '% Humedad',
        'eje_x': '% Humedad',
        'titulo': 'Relación entre la humedad del suelo y la abundancia bacteriana cultivable'
    },
    {
        'columna': '% COS',
        'eje_x': '% COS',
        'titulo': 'Relación entre el carbono orgánico del suelo y la abundancia bacteriana cultivable'
    }
]

alfa = 0.05

# ==============================================================================
# 2. CÁLCULO DE CORRELACIÓN DE SPEARMAN
# ==============================================================================
filas_spearman = []

for config in config_variables:
    var = config['columna']
    df_par = df[[var, 'Log_Control_UFC']].dropna()
    coef_spearman, p_spearman = stats.spearmanr(df_par[var], df_par['Log_Control_UFC'])

    filas_spearman.append({
        'Variable Edáfica': var,
        'Variable Microbiológica': 'Log10(UFC/g + 1)',
        'Muestras Válidas (N)': len(df_par),
        'Coeficiente Spearman (ρ)': round(coef_spearman, 4),
        'p-valor': round(p_spearman, 4),
        'Significativo (p < 0.05)': 'Sí' if p_spearman < alfa else 'No'
    })

tabla_spearman = pd.DataFrame(filas_spearman)

print("=" * 80)
print("TABLA DE CORRELACIÓN DE SPEARMAN (ESCALA LOGARÍTMICA)")
print("=" * 80)
display(tabla_spearman)

# ==============================================================================
# 3. DIAGRAMAS DE DISPERSIÓN CON FORMATO ESPECÍFICO DE EJE Y TITULOS
# ==============================================================================
sns.set_theme(style="whitegrid")

for config in config_variables:
    var = config['columna']
    titulo_grafico = config['titulo']

    # Filtrar subconjunto de datos no nulos
    df_grafico = df[[columna_chakra, var, 'Log_Control_UFC']].dropna()

    fig, ax = plt.subplots(figsize=(8, 6))

    # Gráfico de dispersión usando ax.scatter
    ax.scatter(
        df_grafico[var],
        df_grafico["Log_Control_UFC"],
        s=90,
        color='#2b5c8f',
        edgecolor='black',
        zorder=3
    )

    # Anotación de las etiquetas de Chakra sobre cada punto
    for _, fila in df_grafico.iterrows():
        etiqueta = str(fila[columna_chakra])
        x_val = fila[var]
        y_val = fila['Log_Control_UFC']

        ax.annotate(
            etiqueta,
            (x_val, y_val),
            textcoords="offset points",
            xytext=(5, 5),
            ha='left',
            fontsize=10,
            fontweight='bold',
            color='#1a1a1a'
        )

    # Configuración de etiquetas de ejes y títulos
    ax.set_title(titulo_grafico, fontsize=13, fontweight='bold', pad=15)
    ax.set_xlabel(config['eje_x'], fontsize=12, labelpad=10)
    ax.set_ylabel(r"$\log_{10}(\mathrm{UFC/g}+1)$", fontsize=12, labelpad=10)

    plt.tight_layout()

    # Guardar gráfico
    nombre_img = f"Scatter_Log_{var.replace('%', 'Pct').replace(' ', '_')}_vs_UFC.png"
    plt.savefig(nombre_img, dpi=300)
    plt.show()

# ==============================================================================
# 4. EXPORTACIÓN A EXCEL
# ==============================================================================
archivo_excel_salida = 'Resultados_Correlacion_Log_UFC.xlsx'

with pd.ExcelWriter(archivo_excel_salida) as writer:
    tabla_spearman.to_excel(writer, sheet_name='Correlacion_Spearman', index=False)

print(f"\n Resultados guardados en: '{archivo_excel_salida}'")

In [ ]:
# ============================================================
# SUELO Y RESISTENCIA FENOTÍPICA (CORREGIDO Y ADAPTADO)
# Mann-Whitney + ajuste de Holm + gráficos de 2 cajas + Excel
# ============================================================

# 1. IMPORTAR LIBRERÍAS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from scipy.stats import mannwhitneyu, PermutationMethod
from statsmodels.stats.multitest import multipletests
from IPython.display import display


# ============================================================
# 2. CONFIGURACIÓN DE COLUMNAS
# ============================================================

COL_CHAKRA = "Chakra"
COL_GRUPO = "Sul_alta_Crecimiento"

VARIABLES = [
    "pH",
    "% Humedad",
    "% COS"
]

NOMBRES_GRAFICOS = {
    "pH": "pH",
    "% Humedad": "Humedad del suelo (%)",
    "% COS": "Carbono orgánico del suelo (%)"
}

NOMBRES_ARCHIVOS = {
    "pH": "pH",
    "% Humedad": "Humedad",
    "% COS": "COS"
}


# ============================================================
# 3. COMPROBAR QUE EL DATAFRAME EXISTA
# ============================================================

if "df" not in globals():
    raise NameError(
        "No existe un DataFrame llamado 'df'. "
        "Primero carga tu base y guárdala como df."
    )

base = df.copy()

# Limpiar espacios en los nombres de las columnas
base.columns = base.columns.astype(str).str.strip()

columnas_requeridas = [
    COL_CHAKRA,
    COL_GRUPO,
    *VARIABLES
]

columnas_faltantes = [
    columna
    for columna in columnas_requeridas
    if columna not in base.columns
]

if columnas_faltantes:
    raise ValueError(
        "Faltan estas columnas en la base:\n"
        f"{columnas_faltantes}\n\n"
        f"Columnas disponibles:\n{base.columns.tolist()}"
    )


# ============================================================
# 4. CONVERTIR LAS VARIABLES DEL SUELO A NÚMEROS (VALORES ORIGINALES)
# ============================================================

for variable in VARIABLES:
    if base[variable].dtype == "object":
        base[variable] = (
            base[variable]
            .astype(str)
            .str.strip()
            .str.replace(",", ".", regex=False)
        )

    base[variable] = pd.to_numeric(
        base[variable],
        errors="coerce"
    )


# ============================================================
# 5. LIMPIAR LA VARIABLE DE CRECIMIENTO
# ============================================================

grupo_numerico = pd.to_numeric(
    base[COL_GRUPO],
    errors="coerce"
)

grupo_texto = (
    base[COL_GRUPO]
    .astype(str)
    .str.strip()
    .str.lower()
)

mapa_texto = {
    "0": 0,
    "0.0": 0,
    "sin crecimiento": 0,
    "sin crecimiento (0)": 0,
    "1": 1,
    "1.0": 1,
    "con crecimiento": 1,
    "con crecimiento (1)": 1
}

grupo_numerico = grupo_numerico.fillna(
    grupo_texto.map(mapa_texto)
)

base[COL_GRUPO] = grupo_numerico

# Filtrar únicamente los valores binarios válidos (0 y 1)
base = base[base[COL_GRUPO].isin([0, 1])].copy()
base[COL_GRUPO] = base[COL_GRUPO].astype(int)

base["Fenotipo"] = base[COL_GRUPO].map({
    0: "Sin crecimiento",
    1: "Con crecimiento"
})


# ============================================================
# 6. COMPROBAR LOS GRUPOS
# ============================================================

orden_grupos = ["Sin crecimiento", "Con crecimiento"]
conteo_grupos = base["Fenotipo"].value_counts().reindex(orden_grupos, fill_value=0)

print("=" * 65)
print("NÚMERO DE MUESTRAS POR GRUPO")
print("=" * 65)
print(conteo_grupos)
print()


# ============================================================
# 7. CONFIGURAR LA PRUEBA POR PERMUTACIONES
# ============================================================

try:
    metodo_permutacion = PermutationMethod(
        n_resamples=np.inf,
        random_state=123
    )
except TypeError:
    metodo_permutacion = PermutationMethod(
        n_resamples=np.inf,
        rng=np.random.default_rng(123)
    )


# ============================================================
# 8. CALCULAR DESCRIPTIVOS Y MANN-WHITNEY
# ============================================================

filas_descriptivos = []
filas_resultados = []

for variable in VARIABLES:

    con_crecimiento = base.loc[base[COL_GRUPO] == 1, variable].dropna()
    sin_crecimiento = base.loc[base[COL_GRUPO] == 0, variable].dropna()

    if len(con_crecimiento) == 0 or len(sin_crecimiento) == 0:
        continue

    # Estadísticos descriptivos
    for nombre_grupo, valores in [("Con crecimiento", con_crecimiento), ("Sin crecimiento", sin_crecimiento)]:
        q1 = valores.quantile(0.25)
        q3 = valores.quantile(0.75)
        filas_descriptivos.append({
            "Variable": variable,
            "Grupo": nombre_grupo,
            "n": len(valores),
            "Mediana": valores.median(),
            "Q1": q1,
            "Q3": q3,
            "RIC": q3 - q1,
            "Mínimo": valores.min(),
            "Máximo": valores.max()
        })

    # Prueba U de Mann-Whitney
    prueba = mannwhitneyu(
        con_crecimiento,
        sin_crecimiento,
        alternative="two-sided",
        method=metodo_permutacion
    )

    n_con = len(con_crecimiento)
    n_sin = len(sin_crecimiento)
    r_biserial = ((2 * prueba.statistic) / (n_con * n_sin)) - 1

    filas_resultados.append({
        "Variable": variable,
        "n_con_crecimiento": n_con,
        "Mediana_con": con_crecimiento.median(),
        "Q1_con": con_crecimiento.quantile(0.25),
        "Q3_con": con_crecimiento.quantile(0.75),
        "n_sin_crecimiento": n_sin,
        "Mediana_sin": sin_crecimiento.median(),
        "Q1_sin": sin_crecimiento.quantile(0.25),
        "Q3_sin": sin_crecimiento.quantile(0.75),
        "U": float(prueba.statistic),
        "p_original": float(prueba.pvalue),
        "r_biserial": float(r_biserial)
    })

descriptivos = pd.DataFrame(filas_descriptivos)
resultados = pd.DataFrame(filas_resultados)


# ============================================================
# 9. CORRECCIÓN DE HOLM E INTERPRETACIONAL
# ============================================================

resultados["p_Holm"] = multipletests(resultados["p_original"], alpha=0.05, method="holm")[1]
resultados["Significativo_original"] = resultados["p_original"] < 0.05
resultados["Significativo_Holm"] = resultados["p_Holm"] < 0.05

def clasificar_efecto(valor):
    magnitud = abs(valor)
    if magnitud < 0.10: return "Muy pequeño"
    elif magnitud < 0.30: return "Pequeño"
    elif magnitud < 0.50: return "Moderado"
    else: return "Grande"

resultados["Magnitud_efecto"] = resultados["r_biserial"].apply(clasificar_efecto)

def interpretar_resultado(fila):
    if fila["p_Holm"] < 0.05:
        return "Diferencia significativa después del ajuste de Holm"
    elif fila["p_original"] < 0.05 and fila["p_Holm"] >= 0.05:
        return "Diferencia sin ajustar; no significativa después de Holm"
    elif fila["p_original"] < 0.10:
        return "Tendencia exploratoria; no significativa"
    else:
        return "No se encontraron diferencias significativas"

resultados["Interpretación"] = resultados.apply(interpretar_resultado, axis=1)

# Tabla final formateada
tabla_final = resultados.copy()
tabla_final["Con crecimiento: mediana [Q1–Q3]"] = tabla_final.apply(
    lambda f: f"{f['Mediana_con']:.3f} [{f['Q1_con']:.3f}–{f['Q3_con']:.3f}]", axis=1
)
tabla_final["Sin crecimiento: mediana [Q1–Q3]"] = tabla_final.apply(
    lambda f: f"{f['Mediana_sin']:.3f} [{f['Q1_sin']:.3f}–{f['Q3_sin']:.3f}]", axis=1
)

tabla_final = tabla_final[[
    "Variable", "n_con_crecimiento", "n_sin_crecimiento",
    "Con crecimiento: mediana [Q1–Q3]", "Sin crecimiento: mediana [Q1–Q3]",
    "U", "p_original", "p_Holm", "r_biserial", "Magnitud_efecto", "Interpretación"
]]

print("=" * 65)
print("RESULTADOS DE MANN-WHITNEY Y CORRECCIÓN DE HOLM")
print("=" * 65)
display(tabla_final)


# ============================================================
# 10. GENERACIÓN DE DIAGRAMAS DE CAJA (SEABORN CORREGIDO)
# ============================================================

carpeta_salida = Path("/content/Resultados_Paso_6")
carpeta_salida.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

# Paleta configurada para aceptar llaves numéricas y texto
paleta_colores = {0: '#d95f02', 1: '#1b9e77', '0': '#d95f02', '1': '#1b9e77'}

for variable in VARIABLES:

    resultado_variable = resultados.loc[resultados["Variable"] == variable]
    if resultado_variable.empty:
        continue
    fila_resultado = resultado_variable.iloc[0]

    df_plot = base[[COL_GRUPO, COL_CHAKRA, variable]].dropna()

    plt.figure(figsize=(7, 6))

    # Diagrama de caja con exactamente 2 cajas
    ax = sns.boxplot(
        data=df_plot,
        x=COL_GRUPO,
        y=variable,
        palette=paleta_colores,
        width=0.4,
        medianprops={"color": "black", "linewidth": 2.5}
    )

    # Superposición de puntos por chakra (Stripplot)
    sns.stripplot(
        data=df_plot,
        x=COL_GRUPO,
        y=variable,
        color="black",
        alpha=0.7,
        jitter=0.08,
        size=6
    )

    # Etiquetas de Chakras sobre cada punto
    for i in range(len(df_plot)):
        row = df_plot.iloc[i]
        pos_x = 0 if row[COL_GRUPO] == 0 else 1
        ax.annotate(
            str(row[COL_CHAKRA]),
            xy=(pos_x, row[variable]),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=9
        )

    # Personalización de Ejes
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Sin crecimiento (0)", "Con crecimiento (1)"], fontsize=11)
    ax.set_xlabel("Crecimiento frente a sulfonamida alta (Sul_alta_Crecimiento)", fontsize=11, labelpad=10)
    ax.set_ylabel(NOMBRES_GRAFICOS[variable], fontsize=11, labelpad=10)
    ax.set_title(f"{NOMBRES_GRAFICOS[variable]} según el Crecimiento Bacteriano", fontsize=12, fontweight='bold', pad=15)

    # Recadro de resumen estadístico dentro del gráfico
    texto_resultado = (
        f"U = {fila_resultado['U']:.1f}\n"
        f"p = {fila_resultado['p_original']:.3f}\n"
        f"p Holm = {fila_resultado['p_Holm']:.3f}\n"
        f"r biserial = {fila_resultado['r_biserial']:.3f}\n"
        f"n = {int(fila_resultado['n_sin_crecimiento'])} vs {int(fila_resultado['n_con_crecimiento'])}"
    )

    ax.text(
        0.03, 0.97, texto_resultado,
        transform=ax.transAxes,
        verticalalignment="top",
        fontsize=10,
        bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.8}
    )

    plt.tight_layout()

    # Guardar gráfico
    ruta_imagen = carpeta_salida / f"Boxplot_{NOMBRES_ARCHIVOS[variable]}.png"
    plt.savefig(ruta_imagen, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Gráfico guardado en: {ruta_imagen}")


# ============================================================
# 11. EXPORTAR RESULTADOS A EXCEL
# ============================================================

ruta_excel = carpeta_salida / "Resultados_Mann_Whitney_Holm_Corregidos.xlsx"

with pd.ExcelWriter(ruta_excel, engine="openpyxl") as writer:
    base.to_excel(writer, sheet_name="Datos_analizados", index=False)
    descriptivos.to_excel(writer, sheet_name="Descriptivos", index=False)
    resultados.to_excel(writer, sheet_name="Resultados_completos", index=False)
    tabla_final.to_excel(writer, sheet_name="Tabla_para_tesis", index=False)

print("\n" + "=" * 65)
print("ANÁLISIS FINALIZADO CON ÉXITO")
print("=" * 65)
print(f" Archivo Excel guardado en:\n{ruta_excel}")

In [ ]:
# ============================================================
# PCA DE pH, HUMEDAD Y COS (ADAPTADO Y CORREGIDO)
# Script optimizado para Google Colab
# ============================================================

# 1. INSTALACIÓN / COMPROBACIÓN DE LIBRERÍAS
try:
    import openpyxl
except ImportError:
    !pip install openpyxl --quiet
    import openpyxl

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from google.colab import files
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from IPython.display import display

# ==============================================================================
# 2. CARGA Y PREPARACIÓN DE DATOS
# ==============================================================================
nombre_archivo = 'Pruebas_Estadísticas_TESIS.xlsx'

try:
    df = pd.read_excel(nombre_archivo)
    print("Archivo cargado correctamente.\n")
except Exception as e:
    print(f"Error al cargar el archivo Excel: {e}")

# Limpieza de espacios en blanco en los nombres de columnas
df.columns = df.columns.str.strip()

# ============================================================
# 3. LIMPIEZA Y COMPROBACIÓN DE COLUMNAS
# ============================================================
df.columns = df.columns.astype(str).str.strip()

columnas_requeridas = ["Chakra", "Sitio", "pH", "% Humedad", "% COS"]
columnas_faltantes = [col for col in columnas_requeridas if col not in df.columns]

if columnas_faltantes:
    raise ValueError(
        f"Faltan las siguientes columnas en tu archivo: {columnas_faltantes}\n"
        f"Columnas detectadas: {df.columns.tolist()}"
    )

variables_pca = ["pH", "% Humedad", "% COS"]

# ============================================================
# 4. CONVERSIÓN NUMÉRICA Y MANEJO DE NULOS
# ============================================================
for variable in variables_pca:
    if df[variable].dtype == "object":
        df[variable] = (
            df[variable]
            .astype(str)
            .str.strip()
            .str.replace(",", ".", regex=False)
        )
    df[variable] = pd.to_numeric(df[variable], errors="coerce")

print("\n Conteo de datos faltantes por variable:")
display(df[variables_pca].isna().sum().to_frame("Datos faltantes"))

filas_incompletas = df[columnas_requeridas].isna().any(axis=1)
if filas_incompletas.any():
    print("\n Las siguientes filas contienen datos incompletos y serán excluidas:")
    display(df.loc[filas_incompletas, columnas_requeridas])

base_pca = df[columnas_requeridas].dropna().copy()

print(f"\n Total de muestras válidas incluidas: {len(base_pca)}")
print(f" Número de sitios representados: {base_pca['Sitio'].nunique()}")

# ============================================================
# 5. ESTANDARIZACIÓN (Z-SCORE)
# ============================================================
X = base_pca[variables_pca].copy()

escalador = StandardScaler()
X_estandarizada = escalador.fit_transform(X)

datos_estandarizados = pd.DataFrame(
    X_estandarizada,
    columns=variables_pca,
    index=base_pca.index
)
datos_estandarizados.insert(0, "Sitio", base_pca["Sitio"].values)
datos_estandarizados.insert(0, "Chakra", base_pca["Chakra"].values)

verificacion = pd.DataFrame({
    "Media estandarizada": pd.DataFrame(X_estandarizada, columns=variables_pca).mean(),
    "Desviación estándar": pd.DataFrame(X_estandarizada, columns=variables_pca).std(ddof=0)
})

print("\n Verificación de la estandarización (Media=0, Std=1):")
display(verificacion.round(4))

# ============================================================
# 6. EJECUCIÓN DEL PCA Y VARIANZA EXPLICADA
# ============================================================
pca = PCA(n_components=3)
componentes = pca.fit_transform(X_estandarizada)

resultados_pca = base_pca[["Chakra", "Sitio"]].copy()
resultados_pca["PC1"] = componentes[:, 0]
resultados_pca["PC2"] = componentes[:, 1]
resultados_pca["PC3"] = componentes[:, 2]

varianza_porcentaje = pca.explained_variance_ratio_ * 100
varianza_acumulada = np.cumsum(varianza_porcentaje)

tabla_varianza = pd.DataFrame({
    "Componente": ["PC1", "PC2", "PC3"],
    "Autovalor": pca.explained_variance_,
    "Varianza explicada (%)": varianza_porcentaje,
    "Varianza acumulada (%)": varianza_acumulada
})

print("\n Varianza explicada por componente:")
display(tabla_varianza.round(4))

# ============================================================
# 7. CARGAS Y CONTRIBUCIONES DE LAS VARIABLES
# ============================================================
cargas = pd.DataFrame(
    pca.components_.T,
    index=variables_pca,
    columns=["PC1", "PC2", "PC3"]
)
cargas.index.name = "Variable"

contribuciones = cargas.pow(2)
contribuciones = (contribuciones.div(contribuciones.sum(axis=0), axis=1)) * 100

print("\n Cargas de las variables (Loadings):")
display(cargas.round(4))

print("\n Contribución porcentual a cada componente (%):")
display(contribuciones.round(2))

# Centroides por sitio
centroides = resultados_pca.groupby("Sitio")[["PC1", "PC2"]].mean().reset_index()

# ============================================================
# 8. GENERACIÓN Y GUARDADO DE GRÁFICOS
# ============================================================
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

# --- GRÁFICO 1: Varianza Explicada ---
fig, ax = plt.subplots(figsize=(8, 5))
componentes_nombre = ["PC1", "PC2", "PC3"]
ax.plot(componentes_nombre, varianza_porcentaje, marker="o", linewidth=2, color="#2b5c8f")

for i, pct in enumerate(varianza_porcentaje):
    ax.annotate(f"{pct:.2f} %", (i, pct), xytext=(0, 8), textcoords="offset points", ha="center", fontweight="bold")

ax.set_xlabel("Componente Principal", fontsize=11)
ax.set_ylabel("Varianza Explicada (%)", fontsize=11)
ax.set_title("Varianza Explicada por los Componentes Principales", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("/content/Grafico_varianza_explicada.png", dpi=300, bbox_inches="tight")
plt.show()

# --- GRÁFICO 2: Distribución de Chakras ---
fig, ax = plt.subplots(figsize=(10, 7))
sitios = resultados_pca["Sitio"].unique()

for sitio in sitios:
    grupo = resultados_pca[resultados_pca["Sitio"] == sitio]
    ax.scatter(grupo["PC1"], grupo["PC2"], s=85, label=sitio, alpha=0.9)
    for _, fila in grupo.iterrows():
        ax.annotate(str(fila["Chakra"]), (fila["PC1"], fila["PC2"]), xytext=(5, 5), textcoords="offset points", fontsize=9)

ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax.axvline(0, color="gray", linewidth=0.8, linestyle="--")
ax.set_xlabel(f"PC1 ({varianza_porcentaje[0]:.2f} %)", fontsize=11)
ax.set_ylabel(f"PC2 ({varianza_porcentaje[1]:.2f} %)", fontsize=11)
ax.set_title("Distribución de las Chakras según Propiedades Edáficas", fontsize=12, fontweight="bold")
ax.legend(title="Sitio de Muestreo", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig("/content/PCA_Chakras_por_sitio.png", dpi=300, bbox_inches="tight")
plt.show()

# --- GRÁFICO 3: Biplot Edáfico ---
fig, ax = plt.subplots(figsize=(10, 7))

for sitio in sitios:
    grupo = resultados_pca[resultados_pca["Sitio"] == sitio]
    ax.scatter(grupo["PC1"], grupo["PC2"], s=85, label=sitio, alpha=0.9)
    for _, fila in grupo.iterrows():
        ax.annotate(str(fila["Chakra"]), (fila["PC1"], fila["PC2"]), xytext=(5, 5), textcoords="offset points", fontsize=9)

# Factor de escala para las flechas
rango_pc1 = resultados_pca["PC1"].max() - resultados_pca["PC1"].min()
rango_pc2 = resultados_pca["PC2"].max() - resultados_pca["PC2"].min()
factor_flechas = 0.35 * min(rango_pc1, rango_pc2)

nombres_cortos = {"pH": "pH", "% Humedad": "Humedad", "% COS": "COS"}

for variable in variables_pca:
    extremo_x = cargas.loc[variable, "PC1"] * factor_flechas
    extremo_y = cargas.loc[variable, "PC2"] * factor_flechas

    ax.arrow(0, 0, extremo_x, extremo_y, head_width=0.07, head_length=0.10, linewidth=1.5, color="#d95f02", length_includes_head=True)
    ax.text(extremo_x * 1.15, extremo_y * 1.15, nombres_cortos[variable], color="#d95f02", fontweight="bold", fontsize=11, ha="center")

ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax.axvline(0, color="gray", linewidth=0.8, linestyle="--")
ax.set_xlabel(f"PC1 ({varianza_porcentaje[0]:.2f} %)", fontsize=11)
ax.set_ylabel(f"PC2 ({varianza_porcentaje[1]:.2f} %)", fontsize=11)
ax.set_title("Biplot PCA: Propiedades Físico-Químicas del Suelo", fontsize=12, fontweight="bold")
ax.legend(title="Sitio de Muestreo", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig("/content/Biplot_PCA_fisicoquimico.png", dpi=300, bbox_inches="tight")
plt.show()

# ============================================================
# 9. EXPORTACIÓN A EXCEL Y FINALIZACIÓN
# ============================================================
ruta_excel = "/content/Resultados_PCA_Fisicoquimico.xlsx"

with pd.ExcelWriter(ruta_excel, engine="openpyxl") as writer:
    base_pca.to_excel(writer, sheet_name="Datos_utilizados", index=False)
    datos_estandarizados.to_excel(writer, sheet_name="Datos_estandarizados", index=False)
    resultados_pca.to_excel(writer, sheet_name="Coordenadas_chakras", index=False)
    tabla_varianza.to_excel(writer, sheet_name="Varianza_explicada", index=False)
    cargas.to_excel(writer, sheet_name="Cargas_variables")
    contribuciones.to_excel(writer, sheet_name="Contribuciones")
    centroides.to_excel(writer, sheet_name="Centroides_sitios", index=False)

varianza_pc1_pc2 = varianza_porcentaje[0] + varianza_porcentaje[1]

print("\n" + "=" * 60)
print(" PCA FINALIZADO CON ÉXITO")
print("=" * 60)
print(f"• PC1 explica: {varianza_porcentaje[0]:.2f} %")
print(f"• PC2 explica: {varianza_porcentaje[1]:.2f} %")
print(f"• Acumulado (PC1 + PC2): {varianza_pc1_pc2:.2f} %")
print("\n Archivos generados en tu entorno de Colab:")
print(" - /content/Grafico_varianza_explicada.png")
print(" - /content/PCA_Chakras_por_sitio.png")
print(" - /content/Biplot_PCA_fisicoquimico.png")
print(f" - {ruta_excel}")

# Descarga automática del archivo Excel
files.download(ruta_excel)